# Georgia ITS: extract the expansion project list

Start with the 14-page project list prepared in `02_georgia_power_pdf_prep.ipynb`. This notebook reads the table into pandas, joins wrapped names, adds descriptions from the other five PDFs, checks the result, and saves CSVs.

The list covers multiple ITS sponsors, not just Georgia Power. Keep all rows and their sponsor codes. Costs marked `REDACTED` remain text, not zero or estimated amounts.

We use the existing PyMuPDF library. Its `find_tables()` method detects cells from table grid lines, and `extract()` returns those cells as Python lists. Splitting plain page text is an alternative, but loses column boundaries.

In [1]:
import fitz  # PyMuPDF
import pandas as pd
import re

from gridlock.paths import RAW_DATA_DIR, PROCESSED_DATA_DIR

PDF_PATH = RAW_DATA_DIR / "georgia_power" / "2025_irp" / "selected" / "01_expansion_plan_projects.pdf"
OUTPUT_DIR = PROCESSED_DATA_DIR / "georgia_power" / "2025_irp"
SOURCE_URL = "https://psc.ga.gov/search/facts-document/?documentId=221233"

if not PDF_PATH.exists():
    raise FileNotFoundError("The selected project-list PDF is missing: " + str(PDF_PATH))
print(PDF_PATH)

C:\Users\sebas\PycharmProjects\Git\Project_secret_project\data\raw\georgia_power\2025_irp\selected\01_expansion_plan_projects.pdf


## 1. Inspect the first table
The PDF detector splits the project-name area into three columns (indexes 3, 4, and 5). It also splits some wrapped names into extra rows. Print a few rows first so the cleanup below is easy to follow.

In [2]:
with fitz.open(PDF_PATH) as document:
    tables = document[0].find_tables().tables
    if len(tables) != 1:
        raise ValueError("Expected one project table on the first page.")
    first_rows = tables[0].extract()
    for row in first_rows[:10]:
        print(row)

Consider using the pymupdf_layout package for a greatly improved page layout analysis.


['', '', '', '', None, None, '', '', '', '', '', '', '']
[None, None, 'TEAMS', None, None, None, 'Need Date', 'Project', None, None, 'Estimated Cost -', 'Estimated Cost -', None]
['Zone', 'Year', None, 'Project Name', None, None, None, None, 'Estimated Cost - GPC', 'Estimated Cost - GTC', None, None, 'Totals']
[None, None, 'Number', None, None, None, '2024', 'Sponsor', None, None, 'MEAG', 'DU', None]
['', '', None, '', None, None, None, None, '', '', None, None, '']
[None, None, '', None, None, None, '', '', None, None, '', '', None]
['219', '2025', '19523', '', 'SAV: CC - HYUNDAI MOTORS', '', '1/1/2025', 'SAV', 'REDACTED', 'REDACTED', 'REDACTED', 'REDACTED', 'REDACTED']
[None, None, None, None, 'SAVANNAH AKA. PROJECT EA', None, None, None, None, None, None, None, None]
['212', '2025', '18670', 'GTC: BANKS CROSSING -\nPOND FORK 115 KV', None, None, '5/1/2025', 'GTC', 'REDACTED', 'REDACTED', 'REDACTED', 'REDACTED', 'REDACTED']
['216', '2025', '18492', '', 'MITCHELL - NORTH TIFTON', '', 

## 2. Read projects and join wrapped names
A numeric TEAMS number starts a project. A row containing only name text continues the previous project on that page. Skip repeated headers, empty rows, and the final total. Stop on unexpected rows instead of silently dropping data.

`source_pdf_page` is the page in the smaller PDF. `source_plan_page` comes from the printed page label. Both help us check any row against its source.

Set `state="GA"` for this Georgia planning dataset; retain that column when adding descriptions and exporting.

In [3]:
records = []
page_counts = []

with fitz.open(PDF_PATH) as document:
    for page_number, page in enumerate(document, start=1):
        label = re.search(r"Page\s+(\d+)\s+of\s+304", page.get_text("text"))
        if not label:
            raise ValueError("Missing printed plan page label.")
        tables = page.find_tables().tables
        if len(tables) != 1 or tables[0].col_count != 13:
            raise ValueError("Unexpected table layout on PDF page " + str(page_number))

        page_records = []
        for row in tables[0].extract():
            cells = []
            for cell in row:
                cells.append(" ".join((cell or "").split()))
            if not any(cells):
                continue
            if "TEAMS" in cells or "Zone" in cells or "Number" in cells or cells[0] == "Total":
                continue

            name = " ".join(cells[3:6]).strip()
            name = " ".join(name.split())
            if cells[2].isdigit():
                page_records.append({
                    "state": "GA",
                    "zone": cells[0],
                    "plan_year": cells[1],
                    "project_id": cells[2],
                    "project_name": name,
                    "need_date_raw": cells[6],
                    "project_sponsor": cells[7],
                    "estimated_cost_gpc_raw": cells[8],
                    "estimated_cost_gtc_raw": cells[9],
                    "estimated_cost_meag_raw": cells[10],
                    "estimated_cost_du_raw": cells[11],
                    "estimated_cost_total_raw": cells[12],
                    "source_file": PDF_PATH.name,
                    "source_pdf_page": page_number,
                    "source_plan_page": int(label.group(1)),
                    "source_url": SOURCE_URL,
                })
            elif name and not any(cells[:3]) and not any(cells[6:]) and page_records:
                page_records[-1]["project_name"] += " " + name
            else:
                raise ValueError(f"Unrecognized row on PDF page {page_number}: {cells}")

        if not page_records:
            raise ValueError("No projects found on PDF page " + str(page_number))
        records.extend(page_records)
        page_counts.append({"source_pdf_page": page_number, "projects": len(page_records)})

df_georgia = pd.DataFrame(records)
print("Project entries extracted:", len(df_georgia))
df_georgia.head(10)

Project entries extracted: 208


,state,zone,plan_year,project_id,project_name,need_date_raw,project_sponsor,estimated_cost_gpc_raw,estimated_cost_gtc_raw,estimated_cost_meag_raw,estimated_cost_du_raw,estimated_cost_total_raw,source_file,source_pdf_page,source_plan_page,source_url
0,GA,219,2025,19523,SAV: CC - HYUNDAI MOTORS SAVANNAH AKA. PROJECT EA,1/1/2025,SAV,REDACTED,REDACTED,REDACTED,REDACTED,REDACTED,01_expansion_plan_projects.pdf,1,7,https://psc.ga.gov/search/facts-document/?docu...
1,GA,212,2025,18670,GTC: BANKS CROSSING - POND FORK 115 KV,5/1/2025,GTC,REDACTED,REDACTED,REDACTED,REDACTED,REDACTED,01_expansion_plan_projects.pdf,1,7,https://psc.ga.gov/search/facts-document/?docu...
2,GA,216,2025,18492,MITCHELL - NORTH TIFTON 230KV RECONDUCTOR,5/1/2025,GPC,REDACTED,REDACTED,REDACTED,REDACTED,REDACTED,01_expansion_plan_projects.pdf,1,7,https://psc.ga.gov/search/facts-document/?docu...
3,GA,206,2025,19676,ADAMSVILLE - JACK MCDONOUGH 230KV LINE REBUILD,6/1/2025,GPC,REDACTED,REDACTED,REDACTED,REDACTED,REDACTED,01_expansion_plan_projects.pdf,1,7,https://psc.ga.gov/search/facts-document/?docu...
4,GA,214,2025,18800,ECHECONNEE-WELLSTON 115KV REBUILD,6/1/2025,GPC,REDACTED,REDACTED,REDACTED,REDACTED,REDACTED,01_expansion_plan_projects.pdf,1,7,https://psc.ga.gov/search/facts-document/?docu...
5,GA,211,2025,19187,GRID - BREMEN - CROOKED CREEK (APC) 115 KV PRO...,6/1/2025,GPC,REDACTED,REDACTED,REDACTED,REDACTED,REDACTED,01_expansion_plan_projects.pdf,1,7,https://psc.ga.gov/search/facts-document/?docu...
6,GA,214,2025,18153,GTC: BONAIRE PRI- ECHECONNEE 115 KV PARTIAL RE...,6/1/2025,GTC,REDACTED,REDACTED,REDACTED,REDACTED,REDACTED,01_expansion_plan_projects.pdf,1,7,https://psc.ga.gov/search/facts-document/?docu...
7,GA,218,2025,20342,GTC: CAMDEN INDUSTRIAL PARK 230/115KV NEW SUBS...,6/1/2025,GTC,REDACTED,REDACTED,REDACTED,REDACTED,REDACTED,01_expansion_plan_projects.pdf,1,7,https://psc.ga.gov/search/facts-document/?docu...
8,GA,214,2025,20590,GTC: EATONTON PRIMARY - LICK CREEK 115KV LINE ...,6/1/2025,GTC,REDACTED,REDACTED,REDACTED,REDACTED,REDACTED,01_expansion_plan_projects.pdf,1,7,https://psc.ga.gov/search/facts-document/?docu...
9,GA,218,2025,11821,JESUP - LUDOWICI PRIMARY 115KV REBUILD,6/1/2025,GPC,REDACTED,REDACTED,REDACTED,REDACTED,REDACTED,01_expansion_plan_projects.pdf,1,7,https://psc.ga.gov/search/facts-document/?docu...


## 3. Convert dates and check the data
Preserve the original need date beside the parsed date. A need date is not a construction start date. Keep project IDs as strings. Duplicate IDs and invalid dates are displayed for review, never removed automatically.

In [4]:
df_georgia["plan_year"] = pd.to_numeric(df_georgia["plan_year"], errors="raise").astype(int)
df_georgia["need_date"] = pd.to_datetime(df_georgia["need_date_raw"], format="%m/%d/%Y", errors="coerce")

for column in ["project_id", "project_name", "zone", "project_sponsor", "need_date_raw"]:
    assert df_georgia[column].notna().all(), column
    assert df_georgia[column].str.strip().ne("").all(), column
assert len(df_georgia) == sum(item["projects"] for item in page_counts)

invalid_dates = df_georgia[df_georgia["need_date"].isna()]
duplicate_ids = df_georgia[df_georgia["project_id"].duplicated(keep=False)]
print("Invalid need dates:", len(invalid_dates))
print("Rows with duplicate project IDs:", len(duplicate_ids))
print(invalid_dates[["project_id", "need_date_raw", "source_pdf_page"]].to_string(index=False))
print(duplicate_ids[["project_id", "project_name", "source_pdf_page"]].to_string(index=False))
pd.DataFrame(page_counts)

Invalid need dates: 0
Rows with duplicate project IDs: 0
Empty DataFrame
Columns: [project_id, need_date_raw, source_pdf_page]
Index: []
Empty DataFrame
Columns: [project_id, project_name, source_pdf_page]
Index: []


,source_pdf_page,projects
0,1,13
1,2,15
2,3,15
3,4,15
4,5,15
5,6,16
6,7,15
7,8,16
8,9,17
9,10,17


## 4. Inspect sponsors and the Georgia Power subset
Keep the complete ITS list as `df_georgia`. `df_gpc` selects only rows whose source sponsor code is `GPC`; it does not infer ownership for other sponsors.

In [5]:
print(df_georgia["project_sponsor"].value_counts().to_string())
df_gpc = df_georgia[df_georgia["project_sponsor"] == "GPC"].copy()
print("GPC-sponsored entries:", len(df_gpc))
df_gpc[["project_id", "project_name", "plan_year", "need_date", "source_plan_page"]].head(10)

project_sponsor
GPC     122
GTC      54
SAV      16
MEAG     14
DU        2
GPC-sponsored entries: 122


,project_id,project_name,plan_year,need_date,source_plan_page
2,18492,MITCHELL - NORTH TIFTON 230KV RECONDUCTOR,2025,2025-05-01,7
3,19676,ADAMSVILLE - JACK MCDONOUGH 230KV LINE REBUILD,2025,2025-06-01,7
4,18800,ECHECONNEE-WELLSTON 115KV REBUILD,2025,2025-06-01,7
5,19187,GRID - BREMEN - CROOKED CREEK (APC) 115 KV PRO...,2025,2025-06-01,7
9,11821,JESUP - LUDOWICI PRIMARY 115KV REBUILD,2025,2025-06-01,7
12,20464,NORCROSS 230KV BUS 1-3 SERIES BUS TIE BREAKER ...,2025,2025-06-01,7
13,19631,PINE GROVE PRIMARY 115 KV DUAL STAGE CAPACITOR...,2025,2025-06-01,8
15,20466,SMART VALVE INSTALLATION,2025,2025-06-01,8
16,21046,THALMANN AND COLERAIN 23O KV LINE RELAY PANEL ...,2025,2025-06-01,8
17,20431,VILLA RICA LOW SIDE BREAKER,2025,2025-06-01,8


## 5. Extract descriptions from the four detail sections
`get_text("text", sort=True)` reads text in page order. Without sorting, this PDF places descriptions after the cost fields in its extracted text.

Find each TEAMS header, then read only the text between **Description** and **Supporting Statement**. Preserve source spelling and redaction markers. The first page of each PDF is a section introduction. One other page only continues the preceding project's costs; check it explicitly rather than silently skipping an unknown layout.

In [6]:
DETAIL_FILES = [
    ("02_stability_projects.pdf", "Stability"),
    ("03_short_circuit_projects.pdf", "Short circuit"),
    ("04_interface_transfer_projects.pdf", "Interface transfer capability"),
    ("05_steady_state_projects.pdf", "Steady state"),
]
description_records = []
skipped_pages = []

for filename, section in DETAIL_FILES:
    with fitz.open(PDF_PATH.parent / filename) as document:
        for page_number, page in enumerate(document, start=1):
            text = page.get_text("text", sort=True)
            project = re.search(r"^\s*Teams\s*#\s*(\d+)\s*$", text, re.I | re.M)
            if not project:
                if page_number == 1 and "Project Details" in text:
                    reason = "Section introduction"
                elif "Estimated Cost" in text and "ITS Assigned designation" in text and "Description" not in text:
                    reason = "Cost continuation; no description on this page"
                else:
                    raise ValueError(f"Unexpected page without a TEAMS header: {filename}, page {page_number}")
                skipped_pages.append({"file": filename, "page": page_number, "reason": reason})
                continue

            description = re.search(
                r"^\s*Description\s*\n(.*?)^\s*Supporting Statement\s*$",
                text, re.S | re.M,
            )
            label = re.search(r"Page\s+(\d+)\s+of\s+304", text)
            if not description or not description.group(1).strip() or not label:
                raise ValueError(f"Missing description or page label: {filename}, page {page_number}")
            description_records.append({
                "project_id": project.group(1),
                "description": " ".join(description.group(1).split()),
                "description_section": section,
                "description_source_file": filename,
                "description_pdf_page": page_number,
                "description_plan_page": int(label.group(1)),
            })

df_descriptions = pd.DataFrame(description_records)
assert df_descriptions["project_id"].is_unique, "Duplicate description IDs need review"
print("Descriptions extracted:", len(df_descriptions))
print(pd.DataFrame(skipped_pages).to_string(index=False))
df_descriptions.head()

Descriptions extracted: 208
                              file  page                                         reason
         02_stability_projects.pdf     1                           Section introduction
     03_short_circuit_projects.pdf     1                           Section introduction
04_interface_transfer_projects.pdf     1                           Section introduction
      05_steady_state_projects.pdf     1                           Section introduction
      05_steady_state_projects.pdf    73 Cost continuation; no description on this page


,project_id,description,description_section,description_source_file,description_pdf_page,description_plan_page
0,21046,Relay protection modifications at GPC's Thalma...,Stability,02_stability_projects.pdf,2,44
1,20326,Install a + 150 / - 150 MVAR turnkey STATCOM s...,Stability,02_stability_projects.pdf,3,45
2,17075,Relay modernization with conversion of the 230...,Stability,02_stability_projects.pdf,4,46
3,20491,"Upgrade protection on 230kV buses, 115kV buses...",Stability,02_stability_projects.pdf,5,47
4,20474,"Upgrade protection on 230kV bus, all 115kV bus...",Stability,02_stability_projects.pdf,6,48


## 6. Read the strategic executive summaries
These four projects also appear in the regular project details. Keep their longer executive summaries in a separate `strategic_summary` column instead of replacing their descriptions.

Only a standalone TEAMS header starts a project. TEAMS IDs mentioned inside paragraphs refer to related projects and must not create extra rows. The executive summary ends at the first figure caption or next section heading. Later pages contain supporting analysis, tables, and figures; the source range records where the full strategic discussion can be read.

In [7]:
STRATEGIC_FILE = "06_strategic_projects.pdf"
strategic_records = []

with fitz.open(PDF_PATH.parent / STRATEGIC_FILE) as document:
    for page_number, page in enumerate(document, start=1):
        text = page.get_text("text", sort=True)
        label = re.search(r"Page\s+(\d+)\s+of\s+304", text)
        if not label:
            raise ValueError("Missing strategic plan page label")
        project = re.search(r"^\s*TEAMS\s+(\d+)\s*$", text, re.I | re.M)
        if project:
            summary = re.search(
                r"^\s*Executive Summary\s*\n(.*?)^\s*(?:Figure\s+\d+|Compliance Statement|Background and Problem Description)",
                text, re.S | re.M,
            )
            if not summary or not summary.group(1).strip():
                raise ValueError("Could not read strategic summary on page " + str(page_number))
            strategic_records.append({
                "project_id": project.group(1),
                "strategic_summary": " ".join(summary.group(1).split()),
                "strategic_source_file": STRATEGIC_FILE,
                "strategic_pdf_first_page": page_number,
                "strategic_plan_first_page": int(label.group(1)),
            })
        if not strategic_records:
            raise ValueError("Strategic PDF starts without a project header")
        strategic_records[-1]["strategic_pdf_last_page"] = page_number
        strategic_records[-1]["strategic_plan_last_page"] = int(label.group(1))

df_strategic = pd.DataFrame(strategic_records)
assert df_strategic["project_id"].is_unique, "Duplicate strategic IDs need review"
print("Strategic summaries extracted:", len(df_strategic))
df_strategic[["project_id", "strategic_plan_first_page", "strategic_plan_last_page"]]

Strategic summaries extracted: 4


,project_id,strategic_plan_first_page,strategic_plan_last_page
0,21062,272,276
1,21063,277,280
2,20756,281,284
3,09661,285,297


## 7. Match descriptions to the project list
A pandas `merge` matches rows using `project_id`. `how="left"` keeps every project-list row. `validate="one_to_one"` stops accidental duplication if an ID appears twice.

Start from `records` each time so this cell can be rerun without duplicating columns. Print unmatched IDs in either direction, and stop the export if a description is missing. Strategic summaries are optional because only four projects have that extra section. Original project names, dates, and sponsor codes remain unchanged.

In [8]:
df_georgia = pd.DataFrame(records)
df_georgia["plan_year"] = pd.to_numeric(df_georgia["plan_year"], errors="raise").astype(int)
df_georgia["need_date"] = pd.to_datetime(df_georgia["need_date_raw"], format="%m/%d/%Y", errors="coerce")

missing = df_georgia[~df_georgia["project_id"].isin(df_descriptions["project_id"])]
extra = df_descriptions[~df_descriptions["project_id"].isin(df_georgia["project_id"])]
extra_strategic = df_strategic[~df_strategic["project_id"].isin(df_georgia["project_id"])]
print("List IDs without descriptions:", missing["project_id"].tolist())
print("Description IDs outside the list:", extra["project_id"].tolist())
print("Strategic IDs outside the list:", extra_strategic["project_id"].tolist())
assert missing.empty and extra.empty and extra_strategic.empty, "Unmatched IDs need review before export"

df_georgia = df_georgia.merge(df_descriptions, on="project_id", how="left", validate="one_to_one")
df_georgia = df_georgia.merge(df_strategic, on="project_id", how="left", validate="one_to_one")
assert len(df_georgia) == len(records)
assert df_georgia["description"].notna().all()
for column in ["strategic_pdf_first_page", "strategic_pdf_last_page", "strategic_plan_first_page", "strategic_plan_last_page"]:
    df_georgia[column] = df_georgia[column].astype("Int64")

df_gpc = df_georgia[df_georgia["project_sponsor"] == "GPC"].copy()
print("ITS projects with descriptions:", len(df_georgia))
print("GPC projects with descriptions:", len(df_gpc))
print("Projects with additional strategic summaries:", df_georgia["strategic_summary"].notna().sum())
df_georgia[["project_id", "project_name", "description", "description_plan_page"]].head()

List IDs without descriptions: []
Description IDs outside the list: []
Strategic IDs outside the list: []
ITS projects with descriptions: 208
GPC projects with descriptions: 122
Projects with additional strategic summaries: 4


,project_id,project_name,description,description_plan_page
0,19523,SAV: CC - HYUNDAI MOTORS SAVANNAH AKA. PROJECT EA,Construct a new Hyundai 230kV substation with ...,61
1,18670,GTC: BANKS CROSSING - POND FORK 115 KV,GTC will build a new 115kV line from McClure I...,62
2,18492,MITCHELL - NORTH TIFTON 230KV RECONDUCTOR,Rebuild 35.21 miles of the Mitchell - North Ti...,63
3,19676,ADAMSVILLE - JACK MCDONOUGH 230KV LINE REBUILD,Rebuild the entire 6.1 mile Adamsville - Jack ...,64
4,18800,ECHECONNEE-WELLSTON 115KV REBUILD,Rebuild the South Warner Robins to Wellston se...,65


## 8. Save the enriched project lists
Update the same full ITS and GPC-only CSVs with descriptions, strategic summaries, and their source page references. No additional project CSVs are created. Source PDFs remain unchanged.

When loading either CSV later, use `pd.read_csv(path, dtype={"project_id": "string"})` to retain leading zeros in TEAMS IDs.

In [9]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
ALL_PROJECTS_PATH = OUTPUT_DIR / "georgia_its_project_list.csv"
GPC_PROJECTS_PATH = OUTPUT_DIR / "georgia_power_project_list.csv"

df_georgia.to_csv(ALL_PROJECTS_PATH, index=False, date_format="%Y-%m-%d")
df_gpc.to_csv(GPC_PROJECTS_PATH, index=False, date_format="%Y-%m-%d")

saved = pd.read_csv(ALL_PROJECTS_PATH, dtype={"project_id": "string"})
assert saved["project_id"].tolist() == df_georgia["project_id"].tolist()
print(f"Saved {len(df_georgia)} ITS entries: {ALL_PROJECTS_PATH}")
print(f"Saved {len(df_gpc)} GPC entries: {GPC_PROJECTS_PATH}")

Saved 208 ITS entries: C:\Users\sebas\PycharmProjects\Git\Project_secret_project\data\processed\georgia_power\2025_irp\georgia_its_project_list.csv
Saved 122 GPC entries: C:\Users\sebas\PycharmProjects\Git\Project_secret_project\data\processed\georgia_power\2025_irp\georgia_power_project_list.csv
